# Create a panel for all historical companies in Denmark using Danish CVR

This notebook demonstrates how to get all companies in Denmark, active and disolved, and create a panel dataset. 

Basically, we are imitating https://cvrapi.dk/ 

## The overall steps coded below are:

### Download the all the data using the python scripts in this repo and store them as `parquet` files.

**IMPORTANT**: If you have not downloaded the data you will need to run the `data_extraction` scripts. This step take some time to request the data (API request limits) and requires a fairly good machine (need to be able to merge into 2M rows parquet files).

The strip download all Danish companies with their full history at a given founding year. For example: `python data_extraction/src/virksomhed_api_call.py --founding-years 1991 2000` downloads all the companies founded from 1991 to 2000. All years included.

The oldest CVR in Denmark is University of Copenhagen at  1300, but there are no records until 1735 in the API - so we start from companies founded from 1700 onwards. 

Fell free to check the source code at `data_extraction/src/virksomhed_api_call.py`. 


### Get from the `main` dataset: 

```
    "Vrvirksomhed_virksomhedMetadata_stiftelsesDato": "founded",
    "Vrvirksomhed_cvrNummer": "cvr",
    "Vrvirksomhed_virksomhedMetadata_nyesteNavn_navn": "name",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_vejnavn": "address_1",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_husnummerFra": "address_2",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_postnummer": "zipcode",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_postdistrikt": "city",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_bynavn": "cityname",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_periode_gyldigFra": "startdate",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_periode_gyldigTil": "enddate", 
    "Vrvirksomhed_virksomhedMetadata_nyesteAarsbeskaeftigelse_antalAnsatte": "employees",
    "Vrvirksomhed_virksomhedMetadata_nyesteHovedbranche_branchekode": "industrycode",
    "Vrvirksomhed_virksomhedMetadata_nyesteHovedbranche_branchetekst": "industrydesc",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_virksomhedsformkode": "companycode",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_langBeskrivelse": "companydesc", 
```

### Important: founding date and start date can be different.

- `startdate` and `enddate` only reflects the latest production unit (PO) of the CVR.

- Example: John starts a shoe company in 2000 (PO 1), closes it in 2023 and later  he uses the same CVR to start an AI company (PO 2) in 2024. The `foundingdate` will be 2000 (1st PO) and the `startdate` 2024 (latest PO). 
  
- Companies can re-open at different times using the same CVR number, move from one city to another (e.g. https://cvrapi.dk/api?search=12397399&country=dk), change their legal structure, shut down for some years (e.g. https://cvrapi.dk/api?search=10000173&country=dk) , etc. In any of these cases the companies will have same CVR but different PO.
The data reflects the latest PO available as of 31-12-2025. 

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

pd.set_option('display.max_columns', None)
load_dotenv()

PROJECT_HOME_PATH = os.getenv("PROJECT_HOME_PATH")
COMPANY_DATA_FOLDER_PATH = os.getenv("COMPANY_DATA_FOLDER_PATH")

os.chdir(PROJECT_HOME_PATH)

# Import translation utilities
from utils.translations import COLUMN_TRANSLATIONS, VALUE_TRANSLATIONS

## 1. Download the all the data using the python scripts in this repo and store them as `parquet` files.

In [2]:
# Expect hours downloading data
#!python data_extraction/src/virksomhed_api_call.py --founding-years 1700 1990
#!python data_extraction/src/virksomhed_api_call.py --founding-years 1991 2000
#!python data_extraction/src/virksomhed_api_call.py --founding-years 2001 2010
#!python data_extraction/src/virksomhed_api_call.py --founding-years 2011 2020
#!python data_extraction/src/virksomhed_api_call.py --founding-years 2021 2025


## 2. Get from the `main` dataset the last name and founding date

In [184]:
fields = [
    "Vrvirksomhed_virksomhedMetadata_stiftelsesDato",
    "Vrvirksomhed_cvrNummer",
    "Vrvirksomhed_virksomhedMetadata_nyesteNavn_navn",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_vejnavn",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_husnummerFra",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_postnummer",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_postdistrikt",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_bynavn",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_periode_gyldigFra",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_periode_gyldigTil",
    "Vrvirksomhed_virksomhedMetadata_nyesteAarsbeskaeftigelse_antalAnsatte",
    "Vrvirksomhed_virksomhedMetadata_nyesteHovedbranche_branchekode",
    "Vrvirksomhed_virksomhedMetadata_nyesteHovedbranche_branchetekst",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_virksomhedsformkode",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_langBeskrivelse"
]

translations = {
    "Vrvirksomhed_virksomhedMetadata_stiftelsesDato": "foundingdate",
    "Vrvirksomhed_cvrNummer": "cvr",
    "Vrvirksomhed_virksomhedMetadata_nyesteNavn_navn": "name",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_vejnavn": "address_1",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_husnummerFra": "address_2",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_postnummer": "zipcode",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_postdistrikt": "city",
    "Vrvirksomhed_virksomhedMetadata_nyesteBeliggenhedsadresse_bynavn": "cityname",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_periode_gyldigFra": "startdate",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_periode_gyldigTil": "enddate", 
    "Vrvirksomhed_virksomhedMetadata_nyesteAarsbeskaeftigelse_antalAnsatte": "employees",
    "Vrvirksomhed_virksomhedMetadata_nyesteHovedbranche_branchekode": "industrycode",
    "Vrvirksomhed_virksomhedMetadata_nyesteHovedbranche_branchetekst": "industrydesc",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_virksomhedsformkode": "companycode",
    "Vrvirksomhed_virksomhedMetadata_nyesteVirksomhedsform_langBeskrivelse": "companydesc",  
}

In [188]:
def merge_parquets(dataset_type, fields=None,company_data_folder_path=COMPANY_DATA_FOLDER_PATH):

  df = pd.DataFrame()
  years = ["1700_1990", "1991_2000", "2001_2010", "2011_2020", "2021_2025"]
  for year in years:
    file_path = f"{company_data_folder_path}/virksomhed_founded_{year}_{dataset_type}.parquet"
    df_year = pd.read_parquet(file_path, columns=fields)
    df = pd.concat([df, df_year])

  df.reset_index(drop=True, inplace=True)

  return df

def translate_main(dataframe, translations):
  df = dataframe.rename(columns=translations)
  df = df.sort_values(by="foundingdate")
  return df

In [189]:
main = merge_parquets(dataset_type="main", fields=fields)
main = translate_main(dataframe=main, translations=translations)

In [191]:
main.to_csv("danish_companies.csv", index=False)